# Adding a Prior to the Model

Sometimes the trained potential is not the whole energy: a restraint holding a bond at a
given length, a repulsive term the training data never covered, a bias pushing a
relaxation somewhere it would not go on its own. Such a term belongs to the *model*, not
to the calculator that evaluates it. Once it is part of the energy, the response module
differentiates it along with everything else, and the forces derived from it pick it up
on their own.

SchNetPack's own priors are built exactly this way -- see `ZBLRepulsionEnergy`, the
Coulomb modules, or `HarmonicBond` in `schnetpack.atomistic`. The prior is an output
module that writes its energy under its own `output_key`, and an `Aggregation` module
sums the terms into the total energy.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from copy import deepcopy

from ase.io import read
from ase.optimize import LBFGS

import schnetpack as spk
from schnetpack import properties
from schnetpack.interfaces.ase_interface import SpkCalculator, AtomsConverter
from schnetpack.utils.compatibility import load_model

We load the force field model and wrap it in a `SpkCalculator`, which evaluates one ASE `Atoms` object per call.

In [ ]:
model_path = "../../tests/testdata/md_ethanol.model"

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load model
model = load_model(model_path, device=device)
cutoff = model.representation.cutoff.item()

ENERGY_UNIT = "kcal/mol"
POSITION_UNIT = "Ang"
FMAX = 0.001  # converged once no force exceeds this, in eV/Ang
MAX_STEPS = 1000  # give up after this many optimizer steps


def spk_calculator(model=model_path):
    return SpkCalculator(
        model=model,
        neighbor_list=spk.transform.MatScipyNeighborList(cutoff=cutoff),
        device=device,
        energy_unit=ENERGY_UNIT,
        position_unit=POSITION_UNIT,
    )

The demo below relaxes ethanol's C-O bond starting from a few different conformers.

In [ ]:
input_structure_file = "../../tests/testdata/ethanol_conformers.xyz"

# load the starting structures
conformers = read(input_structure_file, index=":")

In [ ]:
def with_prior(model, prior, prior_key, energy_key=properties.energy):
    """A copy of ``model`` whose energy carries an extra term, forces included.

    The response module differentiates the total energy, so everything contributing to
    that energy has to run ahead of it: the prior and the ``Aggregation`` adding it to the
    model's own prediction are spliced in just before it.
    """
    model = deepcopy(model)
    modules = list(model.output_modules)

    response = next(
        idx
        for idx, module in enumerate(modules)
        if isinstance(module, (spk.atomistic.Forces, spk.atomistic.Response))
    )
    aggregation = spk.atomistic.Aggregation(
        keys=[energy_key, prior_key], output_key=energy_key
    )
    model.output_modules = nn.ModuleList(
        modules[:response] + [prior, aggregation] + modules[response:]
    )

    # the model caches what its modules require and produce, so both have to be redone
    model.collect_derivatives()
    model.collect_outputs()
    return model

Ethanol comes out of the file as `C C O H H H H H H`, so atoms `(0, 2)` are the C-O bond.
It relaxes to about 1.43 Å on its own; we restrain it to 1.8 Å and relax the same
structures twice, once with the plain model and once with the composed one, each
structure one at a time with ASE's `LBFGS`.

`with_prior` re-runs `collect_derivatives` and `collect_outputs`, the two caches
`NeuralNetworkPotential` fills in at construction time. The second one is why
`energy_bond` turns up among the model outputs below, next to `energy` and `forces`.

In [ ]:
BOND = (0, 2)  # the C-O bond of ethanol
BOND_KEY = "energy_bond"

restrained_model = with_prior(
    model,
    spk.atomistic.HarmonicBond(
        atom_pair=BOND,
        bond_length=1.8,  # Angstrom, well beyond the equilibrium C-O distance
        force_constant=50.0,  # eV / Angstrom**2
        energy_unit=ENERGY_UNIT,
        position_unit=POSITION_UNIT,
        output_key=BOND_KEY,
    ),
    prior_key=BOND_KEY,
)

print("output modules:", [type(m).__name__ for m in restrained_model.output_modules])
print("model outputs: ", restrained_model.model_outputs)


def relaxed_bond_lengths(structures, model):
    """Relax every structure with ``model``, one at a time, and report its C-O bond length."""
    calculator = spk_calculator(model)
    lengths = []
    for structure in structures:
        atoms = structure.copy()
        atoms.calc = calculator
        LBFGS(atoms, logfile=None).run(fmax=FMAX, steps=MAX_STEPS)
        lengths.append(atoms.get_distance(*BOND))
    return np.array(lengths)


free_bonds = relaxed_bond_lengths(conformers, model_path)
restrained_bonds = relaxed_bond_lengths(conformers, restrained_model)

print(f"\n{'':<14}{'C-O bond [Ang]':>16}")
for label, bonds in [("unrestrained", free_bonds), ("restrained", restrained_bonds)]:
    print(f"{label:<14}{bonds.mean():>16.3f}")
print(f"{'target':<14}{1.8:>16.3f}")

The restrained bond does not land exactly on the target, and it should not: the restraint
and the model pull against each other, and the relaxation stops where the two balance. At
that point the restraint still pulls outward with `2 * force_constant * (target - d)`, and
that is precisely what the model's own bond force cancels. A stiffer `force_constant`
moves the result closer to the target, a softer one leaves the model more say.

The energy decomposition below makes the same point from the other side. `energy` is the
sum the forces are taken from, and `energy_bond` is the restraint's share of it. One
caveat when reading the numbers: the model's `AddOffsets` postprocessor runs after the
output modules, so the reported total carries the atomref/mean offset while the individual
terms do not.

In [ ]:
# a one-shot conversion this time, so a plain converter with a plain neighbor list --
# nothing is being propagated here, so there is no list worth keeping around
converter = AtomsConverter(
    neighbor_list=spk.transform.MatScipyNeighborList(cutoff=cutoff), device=device
)
plain_results = model(converter(deepcopy(conformers)))
prior_results = restrained_model(converter(deepcopy(conformers)))

print(f"{'structure':>10}{'energy_bond':>14}{'total energy':>14}   [{ENERGY_UNIT}]")
for idx, (bond, total) in enumerate(
    zip(prior_results[BOND_KEY], prior_results[properties.energy])
):
    print(f"{idx:>10}{bond.item():>14.2f}{total.item():>14.2f}")

# and the forces really changed, which is what drove the relaxation above
shift = (
    (prior_results[properties.forces] - plain_results[properties.forces]).abs().max()
)
print(f"\nlargest force change: {shift.item():.1f} {ENERGY_UNIT}/{POSITION_UNIT}")